# ⚡ Road Risk Scoring Model — ONNX Training Pipeline

**Part of:** SafeVisionAI · IIT Madras Road Safety Hackathon 2026
**Output:** `risk_model.onnx` (~21KB) → deployed to `frontend/public/models/`

This notebook trains a **GradientBoosting classifier** to predict real-time road risk
and exports it as ONNX for **in-browser inference** — no server call needed.

---
### 🧠 Model Architecture
| Component | Details |
|-----------|--------|
| Algorithm | GradientBoostingClassifier |
| Input features | 5 (road type, hour, rain, speed limit, prev accidents) |
| Output | Binary: `high_risk` (0 or 1) |
| Export | ONNX via `skl2onnx` |
| Size | ~21KB — loads in milliseconds in browser |

### 🔄 Pipeline
```
Synthetic data generation → GBM training → ONNX conversion → Download
```

> 💡 The model runs entirely client-side in the SafeVisionAI PWA using `onnxruntime-web`.

## 🔧 Step 1 — Install ML Toolkit

Installs the minimum stack needed for training and ONNX export:
- `scikit-learn` — GradientBoostingClassifier
- `skl2onnx` — converts sklearn models to ONNX format
- `pandas` + `numpy` — data generation and manipulation

In [ ]:
# Cell 1 — Install ML Toolkit
!pip install scikit-learn skl2onnx pandas numpy -q
print("✅ Toolkit installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 39.9 MB/s eta 0:00:00
✅ Toolkit installed


## 🏗️ Step 2 — Build Synthetic Training Data

Generates 5,000 synthetic road sensor records matching the live app's data structure:

| Feature | Values | Description |
|---------|--------|-------------|
| `road_type` | 0-3 | NH=0, SH=1, MDR=2, VR=3 |
| `hour` | 0-23 | Hour of day |
| `is_rain` | 0/1 | Weather condition |
| `speed_limit` | 40/60/80/100 | Posted speed (km/h) |
| `prev_accidents` | Poisson(2) | Historical accident count |

**Label logic:** `high_risk = 1` when: Night hours (10pm–4am) + National/State Highway + Raining
This reflects real-world patterns from the India accident dataset.

In [ ]:
# Cell 2 — Build synthetic data matching sensor ingestion structure
import pandas as pd, numpy as np

# Feature: NH=0, SH=1, MDR=2, VR=3
df = pd.DataFrame({
    'road_type': np.random.choice([0,1,2,3], 5000, p=[0.3,0.3,0.25,0.15]),
    'hour': np.random.randint(0,24, 5000),
    'is_rain': np.random.choice([0,1], 5000, p=[0.7,0.3]),
    'speed_limit': np.random.choice([40,60,80,100], 5000),
    'prev_accidents': np.random.poisson(2, 5000)
})

# Synthesize a high-risk label for training (e.g., Night + Rain + Fast highway)
df['high_risk'] = ((df['hour'].isin([22,23,0,1,2,3,4])) & (df['road_type'] <= 1) & (df['is_rain'] == 1)).astype(int)

X = df.drop('high_risk', axis=1)
y = df['high_risk']

print("✅ Data matrix ready. Rows:", len(X))


✅ Data matrix ready. Rows: 5000


## 🎯 Step 3 — Train GradientBoosting Classifier

Trains a GBM with 50 estimators and max depth 4:
- **Fast:** <10 seconds on CPU
- **Accurate:** Handles non-linear risk patterns well
- **Tiny:** Converts to 21KB ONNX — ideal for edge/PWA deployment

In [ ]:
# Cell 3 — Train the GradientBoosting Classifier
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(n_estimators=50, max_depth=4)
model.fit(X, y)
print("✅ Risk AI Training finished")


✅ Risk AI Training finished


## 📦 Step 4 — Export to ONNX & Download

Converts the trained sklearn model to ONNX format using `skl2onnx`:
- **Input:** `FloatTensorType([None, 5])` — batch of 5-feature vectors
- **Output:** Risk probability + binary class label

Download `risk_model.onnx` and place at:
```
frontend/public/models/risk_model.onnx
```

The Next.js PWA loads this at startup and runs inference on each map segment click.

> ✅ Final output: **~21KB** ONNX model — ready for browser deployment

In [ ]:
# Cell 4 — Package as ONNX and Export
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from google.colab import files

initial_type = [('features', FloatTensorType([None, 5]))]
onx = convert_sklearn(model, initial_types=initial_type)

with open('risk_model.onnx', 'wb') as f:
    f.write(onx.SerializeToString())

print(f"✅ Executed. Output size: {len(onx.SerializeToString())//1024} KB")
files.download('risk_model.onnx')


✅ Executed. Output size: 21 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>